# Non-LLM models for title interest

No generative LLM is used in this notebook. Every model is trained on `train.csv` only (competition rule). The
features come from hand-written rules and from frozen pretrained *encoders* (already cached locally, open licenses):

1. **Richer hand-crafted title features** (length, caps, punctuation, numbers, emoji, dates/episode numbers,
   "Official/Trailer/Lyrics" boilerplate, first/second person, …): easy to interpret.
2. **Three frozen text encoders**: `all-mpnet-base-v2` (Apache-2.0), `bert-base-uncased` (Apache-2.0),
   CLIP ViT-H-14 text tower (LAION, MIT) → ridge / SVR / kNN.
3. **Fine-tuned mpnet**: all transformer weights are trained, with a regression head on top.
4. **Ensemble** of a pre-chosen set of the above.

Same 5-fold × 10-repeat CV as the other notebooks. The CV score is the MAE against single-annotator train labels.
The test labels are means of 6 raters, so they are less noisy, and CV MAE is therefore much higher than the
public score (pessimistic). References: constant 1.495 (public 0.848), mpnet SVR from baseline.ipynb 1.466.

**How to read the MAE numbers.** The training labels have a standard deviation of 1.68 on the 1–6 scale.
The constant model (always 3.13) has MAE 1.495, so a model that reaches 1.45 is only about 3 % better.
The ± value is the spread over the 50 CV splits (about 0.17–0.20). Differences of 0.01–0.02 between two models
are therefore mostly noise.

In [1]:
import random, re, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import RepeatedKFold
from sklearn.metrics import mean_absolute_error
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

train = pd.read_csv("data/train.csv"); test = pd.read_csv("data/test.csv"); sample = pd.read_csv("data/sample_submission.csv")
df = pd.concat([train, test], ignore_index=True)
n = len(train); y = train["interest"].values.astype(float)
dom = pd.get_dummies(df["domain"]).values.astype(float)
cv = list(RepeatedKFold(n_splits=5, n_repeats=10, random_state=SEED).split(y))
N_REP = 10
results, oof = {}, {}

def evaluate(name, make_model, X, verbose=True):
    """Score one model with the shared repeated cross-validation (5 folds x 10 repeats = 50 splits).

    make_model: function that builds a NEW untrained model, e.g. lambda: Ridge(alpha=10).
                A fresh model per split guarantees nothing learned on one split leaks into another.
    X:          feature matrix for the 120 training titles (same row order as y).
    For every split: fit on ~96 titles, predict the ~24 held-out titles, clip to the valid range 1-6, compute MAE.
    Prints and stores mean +/- std of the 50 MAEs in `results[name]`.
    Also stores the out-of-fold predictions (shape: 10 repeats x 120 titles) so later cells can test
    shrinking predictions toward the mean, blending models, or correlations without refitting.
    """
    o = np.zeros((N_REP, n)); maes = []
    for k, (a, b) in enumerate(cv):
        p = np.clip(make_model().fit(X[a], y[a]).predict(X[b]), 1, 6)
        o[k // 5, b] = p; maes.append(mean_absolute_error(y[b], p))
    results[name] = dict(mae=np.mean(maes), std=np.std(maes)); oof[name] = o
    if verbose: print(f"{name:45s} MAE {np.mean(maes):.3f} ± {np.std(maes):.3f}")

class Const:
    """Baseline 'model' that ignores the features and always predicts one number (the training mean).
    Every real model has to beat this reference point.
    """
    # remember the mean of the training labels
    def fit(self, X, y): self.c = y.mean(); return self
    # same value for every title
    def predict(self, X): return np.full(len(X), self.c)

evaluate("constant mean", Const, dom[:n])
evaluate("domain ridge", lambda: Ridge(alpha=10), dom[:n])

constant mean                                 MAE 1.495 ± 0.174
domain ridge                                  MAE 1.478 ± 0.181


**Notes on the setup cell**

* `dom` is a one-hot matrix of the domain (article, podcast, reddit, youtube). Each domain has exactly 30 train titles.
* `cv` holds the 50 train/validation splits. All models use the same splits, so their scores can be compared fairly.
* `evaluate()` trains a *new* model on about 96 titles, predicts the other 24, clips to 1–6 and records the MAE.
  It also keeps the out-of-fold predictions (`oof`), which section 4 uses for the ensemble and the shrinkage test.
* `constant mean` (1.495) is the reference. `domain ridge` (1.478) shows that the domain alone helps only a little:
  some domains get slightly higher average scores than others.

## 1. Richer hand-crafted features

In [2]:
MONTHS = r"\b(jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec)[a-z]*\b"
BOILER = r"\b(official|trailer|teaser|lyrics?|lyric video|music video|full episode|live|update|highlights|audio)\b"
EMOJI = re.compile("[\U0001F300-\U0001FAFF\u2600-\u27BF]")

def handcrafted(titles):
    """24 rule-based features per title (version 2 of baseline.ipynb's handcrafted()).

    Length and style: word count, log length, average word length, ALL-CAPS ratio, Title Case ratio.
    Punctuation:      ?, ! count, ellipsis, quotes, separators (| - : dashes), brackets, hashtags, emoji.
    Numbers:          any digit, starts with a digit, year or month name, episode/part numbers.
    Word lists:       boilerplate (official, trailer, lyrics, live...), 'you', first person, Reddit 'TIL',
                      hype words (best, insane, shocking...), how/why/what questions, crime/news words.
    Returns a DataFrame (one row per title). Domain interactions are added in the next cell.
    """
    t = pd.Series(titles).fillna("")
    low = t.str.lower(); w = t.str.split()
    f = pd.DataFrame(index=t.index)
    f["n_words"] = w.str.len()
    f["log_chars"] = np.log1p(t.str.len())
    f["avg_word_len"] = t.str.replace(r"\W", "", regex=True).str.len() / f["n_words"].clip(lower=1)
    f["caps_ratio"] = w.apply(lambda x: np.mean([s.isupper() and len(s) > 1 for s in x]) if x else 0)
    f["title_case_ratio"] = w.apply(lambda x: np.mean([s[:1].isupper() for s in x[1:]]) if len(x) > 1 else 0)
    f["question"] = t.str.contains(r"\?").astype(int)
    f["exclaim"] = t.str.count("!")
    f["ellipsis"] = t.str.contains(r"\.\.|…").astype(int)
    f["quotes"] = t.str.contains("[\"“”']").astype(int)
    f["separators"] = t.str.count(r"\||-|:|–|—")
    f["brackets"] = t.str.count(r"[\(\[]")
    f["hashtags"] = t.str.count("#")
    f["emoji"] = t.apply(lambda s: len(EMOJI.findall(s)))
    f["has_digit"] = t.str.contains(r"\d").astype(int)
    f["starts_digit"] = t.str.match(r"^\d").astype(int)
    f["year_or_date"] = (low.str.contains(r"\b(19|20)\d{2}\b") | low.str.contains(MONTHS)).astype(int)
    f["episode_no"] = low.str.contains(r"\b(ep|episode|part|pt|season|s\d+e\d+)\b\.?\s*\d*").astype(int)
    f["boilerplate"] = low.str.count(BOILER)
    f["you"] = low.str.contains(r"\b(you|your|you're|yourself)\b").astype(int)
    f["first_person"] = low.str.contains(r"\b(i|i'm|we|my|our|me)\b").astype(int)
    f["til"] = low.str.startswith("til").astype(int)
    f["superlative"] = low.str.contains(r"\b(best|worst|most|ever|insane|crazy|amazing|incredible|shocking|secret|every)\b").astype(int)
    f["how_why_what"] = low.str.contains(r"\b(how|why|what|when|who)\b").astype(int)
    f["news_words"] = low.str.contains(r"\b(police|charged|killed|dies|shot|court|arrest|man|woman|officials)\b").astype(int)
    return f

H = handcrafted(df["title"].values)
corr = pd.DataFrame({d: H[:n][train.domain == d].corrwith(train.interest[train.domain == d], method="spearman")
                     for d in ["article", "podcast", "reddit", "youtube"]})
corr.insert(0, "all", H[:n].corrwith(train.interest, method="spearman"))
corr.round(2).sort_values("all")

/var/folders/7s/rgb20vpx2f9_ly3fy0b4x0pw0000gn/T/ipykernel_76606/903924788.py:24: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  f["year_or_date"] = (low.str.contains(r"\b(19|20)\d{2}\b") | low.str.contains(MONTHS)).astype(int)
/var/folders/7s/rgb20vpx2f9_ly3fy0b4x0pw0000gn/T/ipykernel_76606/903924788.py:25: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  f["episode_no"] = low.str.contains(r"\b(ep|episode|part|pt|season|s\d+e\d+)\b\.?\s*\d*").astype(int)
/var/folders/7s/rgb20vpx2f9_ly3fy0b4x0pw0000gn/T/ipykernel_76606/903924788.py:27: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  f["you"] = low.str.contains(r"\b(you|your|you're|yourself)\b").astype(int)
/var/folders/7s/rgb20vpx2f9_ly3fy0b4x0pw0000gn/T/ipykernel_76606/903924788

,all,article,podcast,reddit,youtube
hashtags,-0.21,NaN,-0.31,NaN,-0.20
emoji,-0.17,NaN,NaN,NaN,-0.29
title_case_ratio,-0.14,0.06,-0.02,0.08,0.05
brackets,-0.13,NaN,0.10,0.19,-0.29
quotes,-0.11,0.04,-0.18,0.07,-0.25
how_why_what,-0.08,0.04,-0.15,-0.42,0.16
boilerplate,-0.08,-0.03,-0.33,NaN,0.02
year_or_date,-0.07,-0.20,-0.23,NaN,0.07
episode_no,-0.06,NaN,-0.02,NaN,0.01
exclaim,-0.02,-0.27,-0.05,0.05,0.17


**Reading the correlation table**

Each number is the Spearman correlation between one feature and the interest label: column `all` uses all 120
titles, the other columns use only the 30 titles of one domain. `NaN` means the feature never changes inside that
domain (for example, articles never contain hashtags), so no correlation can be computed.

* With 120 titles, a correlation is clearly different from zero only when |r| > 0.18. With 30 titles the limit is
  about |r| > 0.36. Most values in the table are below these limits and may be noise.
* Over all titles, **hashtags (−0.21)** and **emoji (−0.17)** go with lower interest, and **longer titles
  (n_words +0.19)** go with higher interest.
* The strongest signals exist only inside one domain:
  * podcast: longer titles score higher (n_words +0.46, log_chars +0.40);
  * reddit: questions and how/why/what titles score lower (−0.39, −0.42);
  * youtube: hashtags, emoji and brackets score lower (−0.20 to −0.29).
* The same feature can point in opposite directions in different domains (e.g. `question`: +0.17 for podcast,
  −0.39 for reddit). A single weight per feature cannot express this. That is the motivation for the domain
  interactions in the next cell.

The `UserWarning` lines come from regular expressions with groups inside `str.contains`. They are harmless; the
patterns still match correctly.

In [3]:
Hs = H.values.astype(float)
# domain interactions: each feature gets a per-domain copy (lets e.g. 'news_words' mean different things per domain)
H_int = np.hstack([Hs] + [Hs * dom[:, [j]] for j in range(dom.shape[1])])
for a in (30, 100, 300):
    evaluate(f"handcrafted ridge a={a}", lambda a=a: make_pipeline(StandardScaler(), Ridge(alpha=a)), np.hstack([Hs, dom])[:n])
for a in (100, 300, 1000):
    evaluate(f"handcrafted x domain ridge a={a}", lambda a=a: make_pipeline(StandardScaler(), Ridge(alpha=a)), np.hstack([H_int, dom])[:n])
evaluate("handcrafted extra-trees", lambda: ExtraTreesRegressor(300, min_samples_leaf=8, max_features=0.5, random_state=SEED),
         np.hstack([Hs, dom])[:n])

handcrafted ridge a=30                        MAE 1.493 ± 0.212
handcrafted ridge a=100                       MAE 1.470 ± 0.190


handcrafted ridge a=300                       MAE 1.468 ± 0.178
handcrafted x domain ridge a=100              MAE 1.460 ± 0.205
handcrafted x domain ridge a=300              MAE 1.449 ± 0.185


handcrafted x domain ridge a=1000             MAE 1.461 ± 0.181


handcrafted extra-trees                       MAE 1.521 ± 0.185


**Notes on the hand-crafted models**

* **Domain interactions** (`H_int`): every feature gets four extra copies, one per domain. The copy for reddit is
  the feature value on reddit titles and 0 elsewhere. Example: `question` becomes `question`, `question×article`,
  `question×podcast`, `question×reddit`, `question×youtube`. Ridge can then give a negative weight to questions on
  reddit and a positive weight on podcasts. This gives 24 × 5 = 120 features for only 96 training titles per fold.
* **Strong regularisation is needed.** With so many features and so few titles, ridge needs a large alpha:
  a = 30 gives 1.493, a = 300 gives the best value 1.449. A larger alpha pulls the weights toward 0 and keeps the
  model from memorising the training titles.
* **Extra-trees** (1.521) is worse than the constant model. Tree ensembles need many more samples to find stable splits.
* Result: `handcrafted x domain ridge a=300` (1.449) is the best hand-crafted model, about 0.046 better than the constant.

## 2. Frozen encoders

An **encoder** is a pretrained neural network that turns a text into a fixed list of numbers (an *embedding*).
Titles with similar meaning or style get similar vectors. "Frozen" means the encoder weights are never changed:
the encoder is only used to compute the vectors, and a small model (ridge, SVR or kNN) learns to map them to a score.

The input text is `"<domain>: <title>"`, e.g. `"youtube: I quit my job to travel the world"`.
Three encoders are compared:

* `all-mpnet-base-v2`: trained to place sentences with the same meaning close together (768 numbers).
* `bert-base-uncased`: a general language model; mean pooling of its token vectors (768 numbers).
* CLIP ViT-H-14 text tower: trained to match captions with images, so it may capture how vivid a title is (1024 numbers).

In [4]:
from transformers import AutoTokenizer, AutoModel

def hf_embed(texts, model_name, batch_size=32, max_length=128):
    """Turn each text into one sentence-embedding vector (frozen pretrained transformer, no training).

    Steps per batch:
      1. tokenize: split text into word pieces, pad to equal length (attention_mask = 1 for real tokens, 0 for padding)
      2. run the encoder -> last_hidden_state: one vector per token, shape (batch, tokens, hidden)
      3. mean pooling: average the token vectors, ignoring padding -> one vector per text
      4. L2-normalise to length 1, so only the direction (meaning) matters
    Returns a numpy array of shape (len(texts), hidden_size), e.g. 768 for mpnet/BERT.
    The encoder is deleted afterwards to free GPU memory.
    """
    tok = AutoTokenizer.from_pretrained(model_name)
    enc = AutoModel.from_pretrained(model_name).to(DEVICE).eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            b = tok(list(texts[i:i + batch_size]), padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(DEVICE)
            h = enc(**b).last_hidden_state; m = b["attention_mask"].unsqueeze(-1).float()
            out.append(torch.nn.functional.normalize((h * m).sum(1) / m.sum(1), dim=1).cpu().numpy())
    del enc
    return np.vstack(out)

def clip_text_embed(texts, arch="ViT-H-14", pretrained="laion2b_s32b_b79k"):
    """Embed texts with the TEXT tower of the CLIP ViT-H-14 model (trained on image-caption pairs from LAION-2B).
    CLIP learned which captions match which pictures, so it may capture how vivid or visual a title is.
    Returns L2-normalised vectors (1024 numbers per title).
    """
    import open_clip
    model, _, _ = open_clip.create_model_and_transforms(arch, pretrained=pretrained)
    model = model.to(DEVICE).eval(); tok = open_clip.get_tokenizer(arch)
    out = []
    with torch.no_grad():
        for i in range(0, len(texts), 64):
            e = model.encode_text(tok(list(texts[i:i + 64])).to(DEVICE))
            out.append(torch.nn.functional.normalize(e, dim=1).cpu().numpy())
    del model
    return np.vstack(out)

texts = (df["domain"] + ": " + df["title"]).values
t0 = time.time()
EMB = {
    "mpnet": hf_embed(texts, "sentence-transformers/all-mpnet-base-v2"),
    "bert": hf_embed(texts, "bert-base-uncased"),
    "clipH": clip_text_embed(texts),
}
print({k: v.shape for k, v in EMB.items()}, f"{time.time() - t0:.0f}s")

/Users/home/workspace/School/ML/competition/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1122 01:14:16.354000 76606 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5336.98it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5536.39it/s]


[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'mpnet': (150, 768), 'bert': (150, 768), 'clipH': (150, 1024)} 24s


In [5]:
for k, E in EMB.items():
    X = np.hstack([E, dom])
    for a in (10, 30):
        evaluate(f"{k} ridge a={a}", lambda a=a: Ridge(alpha=a), X[:n])
    for C in (0.3, 1):
        evaluate(f"{k} SVR C={C}", lambda C=C: make_pipeline(StandardScaler(), SVR(C=C, epsilon=0.5)), X[:n])
    evaluate(f"{k} kNN k=30", lambda: KNeighborsRegressor(30, metric="cosine", weights="distance"), E[:n])
X_all = np.hstack([EMB["mpnet"], EMB["clipH"], dom])
evaluate("mpnet+clipH SVR C=0.3", lambda: make_pipeline(StandardScaler(), SVR(C=0.3, epsilon=0.5)), X_all[:n])

mpnet ridge a=10                              MAE 1.481 ± 0.183
mpnet ridge a=30                              MAE 1.480 ± 0.179


mpnet SVR C=0.3                               MAE 1.485 ± 0.173


mpnet SVR C=1                                 MAE 1.473 ± 0.185
mpnet kNN k=30                                MAE 1.480 ± 0.183


bert ridge a=10                               MAE 1.469 ± 0.178
bert ridge a=30                               MAE 1.476 ± 0.177


bert SVR C=0.3                                MAE 1.476 ± 0.170


bert SVR C=1                                  MAE 1.433 ± 0.172
bert kNN k=30                                 MAE 1.445 ± 0.171


clipH ridge a=10                              MAE 1.462 ± 0.180
clipH ridge a=30                              MAE 1.473 ± 0.178


clipH SVR C=0.3                               MAE 1.501 ± 0.172
clipH SVR C=1                                 MAE 1.464 ± 0.182


clipH kNN k=30                                MAE 1.441 ± 0.185


mpnet+clipH SVR C=0.3                         MAE 1.498 ± 0.173


**Notes on the frozen encoders**

* **bert SVR C=1 (1.433)** is the best single model in this notebook. clipH kNN (1.441) and bert kNN (1.445) follow.
* **mpnet** (1.47–1.49) is weaker than bert here, although mpnet is the better general sentence encoder.
  mpnet is trained to put texts with the *same meaning* close together. Interest depends more on style
  (length, tone, clickbait cues), and plain bert vectors may keep more of this style information.
* **Settings**: SVR uses `epsilon=0.5`, so errors smaller than 0.5 are ignored during training. This makes the model
  robust to noisy single-annotator labels. A larger C (1 instead of 0.3) allows a more flexible fit. kNN predicts
  the distance-weighted average score of the 30 most similar training titles (cosine similarity).
* **Combining encoders does not help**: mpnet + clipH together (1.498) is worse than each alone, because the number
  of features (768 + 1024) grows while the number of titles stays at 96.
* **Caution**: 27 models are compared, and the best one is chosen by its CV score. Part of the advantage of the
  winner is luck, so its true error is probably a little higher than 1.433.

## 3. Fine-tuned mpnet (trained neural network)

**Frozen vs fine-tuned.** In section 2 mpnet only produces vectors, and its weights never change. Here *all*
mpnet weights are trained together with a new linear output layer (the *head*, 768 → 1 number), so the network
can learn which parts of a title matter for interest.

Setup: input `"<domain>: <title>"`; mean pooling of the token vectors; MSE loss on the standardised score;
AdamW with learning rate 2e-5 for the pretrained encoder (small, so the pretrained knowledge is not destroyed) and
1e-3 for the new head (larger, because it starts from random weights); 6 epochs, batch size 16. The number of
epochs is fixed: early stopping on the validation fold would leak information and make the CV score too good.
Run on the same 50 folds as every other model.

In [6]:
class FineTuned:
    """Fine-tune the whole mpnet transformer plus a linear head to predict the score directly.

    Unlike the frozen embeddings, every transformer weight is updated. The target is standardised
    ((y - mean) / sd). Two learning rates: small (2e-5) for the pretrained encoder so it is not
    destroyed, larger (1e-3) for the new randomly initialised head. Overfitted: ~96 titles per fold are too few.
    """
    def __init__(self, model_name="sentence-transformers/all-mpnet-base-v2", epochs=6, lr=2e-5, head_lr=1e-3, batch=16, seed=SEED):
        """Store all hyper-parameters as attributes (self.epochs, self.lr, ...)."""
        self.__dict__.update(locals()); del self.__dict__["self"]
    def _pool(self, b):
        """Mean-pool the token vectors of a tokenized batch, ignoring padding -> one vector per title."""
        h = self.enc(**b).last_hidden_state; m = b["attention_mask"].unsqueeze(-1).float()
        return (h * m).sum(1) / m.sum(1)
    def fit(self, X, y):
        """Load a fresh pretrained copy, then train encoder + head with MSE for a fixed number of epochs (fixed seed)."""
        torch.manual_seed(self.seed); g = torch.Generator().manual_seed(self.seed)
        self.tok = AutoTokenizer.from_pretrained(self.model_name)
        self.enc = AutoModel.from_pretrained(self.model_name).to(DEVICE)
        self.head = torch.nn.Linear(self.enc.config.hidden_size, 1).to(DEVICE)
        self.mu, self.sd = y.mean(), y.std()
        yt = torch.tensor((y - self.mu) / self.sd, dtype=torch.float32)
        opt = torch.optim.AdamW([{"params": self.enc.parameters(), "lr": self.lr},
                                 {"params": self.head.parameters(), "lr": self.head_lr}], weight_decay=0.01)
        X = list(X)
        for _ in range(self.epochs):
            self.enc.train()
            for idx in torch.randperm(len(X), generator=g).split(self.batch):
                b = self.tok([X[i] for i in idx], padding=True, truncation=True, max_length=64, return_tensors="pt").to(DEVICE)
                loss = torch.nn.functional.mse_loss(self.head(self._pool(b)).squeeze(1), yt[idx].to(DEVICE))
                opt.zero_grad(); loss.backward(); opt.step()
        return self
    def predict(self, X):
        """Predict in batches, undo the standardisation, and free the model's GPU memory afterwards."""
        self.enc.eval(); out = []
        with torch.no_grad():
            for i in range(0, len(X), 32):
                b = self.tok(list(X[i:i + 32]), padding=True, truncation=True, max_length=64, return_tensors="pt").to(DEVICE)
                out.append(self.head(self._pool(b)).squeeze(1).cpu().numpy())
        p = np.concatenate(out) * self.sd + self.mu
        del self.enc; torch.mps.empty_cache() if DEVICE == "mps" else None
        return p

T = texts.astype(object)
t0 = time.time()
evaluate("fine-tuned mpnet ep=6", lambda: FineTuned(), T[:n])
print(f"{time.time() - t0:.0f}s")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7802.74it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 16444.69it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15747.53it/s]

... (91 outputs left out)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 17310.27it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15866.37it/s]

fine-tuned mpnet ep=6                         MAE 1.500 ± 0.205
436s


**Notes on the fine-tuned model**

* **Result: 1.500**, slightly worse than the constant model (1.495). Fine-tuning does not work with this amount of data.
* **Reason: overfitting.** mpnet has about 110 million weights, but each fold has only about 96 training titles.
  After 6 epochs the network has seen every title 6 times and starts to memorise the labels, so it does not
  learn a general rule. The frozen encoders avoid this problem because only a small model (ridge, SVR, kNN) is trained.
* **Cost**: 436 s for the 50 folds, against a few seconds for every frozen-encoder model.
* The "Loading weights" lines appear because a fresh pretrained copy is loaded for every fold. This is
  intentional: otherwise training from one fold would leak into the next.

## 4. Results, shrinkage and a pre-chosen ensemble

The ensemble is fixed *before* its score is seen: the average of the best model from each family
(hand-crafted, each frozen encoder, fine-tuned). Choosing the members after looking at the ensemble score would
make the CV result too optimistic.

In [7]:
res = pd.DataFrame(results).T.sort_values("mae")
fam = {"handcrafted": [k for k in res.index if k.startswith("handcrafted")],
       "mpnet": [k for k in res.index if k.startswith("mpnet ")], "bert": [k for k in res.index if k.startswith("bert")],
       "clipH": [k for k in res.index if k.startswith("clipH")], "finetuned": [k for k in res.index if k.startswith("fine-tuned")]}
members = [v[0] for v in fam.values() if v]
ens = np.mean([oof[m] for m in members], 0)
results["ensemble (best per family)"] = dict(mae=np.mean([mean_absolute_error(y, ens[r]) for r in range(N_REP)]), std=np.nan)
oof["ensemble (best per family)"] = ens
print("ensemble members:", members)

mu = y.mean(); s_grid = np.round(np.arange(0, 1.51, 0.1), 1)
res = pd.DataFrame(results).T.sort_values("mae")
top = [k for k in res.index if k != "constant mean"][:8]
shrink = pd.DataFrame({k: [np.mean([mean_absolute_error(y, mu + s * (oof[k][r] - mu)) for r in range(N_REP)]) for s in s_grid] for k in top}, index=s_grid)
res["best_s"] = pd.Series({k: shrink[k].idxmin() for k in top}); res["mae_at_best_s"] = pd.Series({k: shrink[k].min() for k in top})
res["oof_corr"] = pd.Series({k: np.mean([np.corrcoef(oof[k][r], y)[0, 1] for r in range(N_REP)]) for k in res.index if k != "constant mean"})
res.round(3)

ensemble members: ['handcrafted x domain ridge a=300', 'mpnet SVR C=1', 'bert SVR C=1', 'clipH kNN k=30', 'fine-tuned mpnet ep=6']


,mae,std,best_s,mae_at_best_s,oof_corr
bert SVR C=1,1.433,0.172,1.4,1.428,0.148
ensemble (best per family),1.439,NaN,1.4,1.431,0.112
clipH kNN k=30,1.441,0.185,1.5,1.431,0.138
bert kNN k=30,1.445,0.171,1.5,1.438,0.131
handcrafted x domain ridge a=300,1.449,0.185,1.0,1.449,0.082
handcrafted x domain ridge a=100,1.460,0.205,0.7,1.448,0.084
handcrafted x domain ridge a=1000,1.461,0.181,1.1,1.461,0.060
clipH ridge a=10,1.462,0.180,1.5,1.459,0.089
clipH SVR C=1,1.464,0.182,NaN,NaN,0.068
handcrafted ridge a=300,1.468,0.178,NaN,NaN,0.039


**How to read the results table**

* `mae`, `std`: mean and spread of the MAE over the 50 CV splits (sorted from best to worst).
* `best_s`, `mae_at_best_s`: **shrinkage test** for the 8 best models. Each prediction p is replaced by
  `mean + s × (p − mean)`. s = 0 gives the constant model, s = 1 keeps the predictions, s > 1 stretches them away
  from the mean. Example with mean 3.13 and p = 3.6: s = 1.5 gives 3.13 + 1.5 × 0.47 = 3.84.
  * Most `best_s` values are 1.4–1.5: the models predict too close to the mean, and stretching helps a little.
  * The gain is small (e.g. bert SVR 1.433 → 1.428, only 0.005) and s is chosen on the same data, so the gain is
    noise-level. The submission therefore keeps s = 1.
* `oof_corr`: Pearson correlation between the out-of-fold predictions and the labels. The best value is only 0.148,
  so all non-LLM models catch only a weak signal.
* **Ensemble** (best model of each family, chosen before its score was seen): 1.439. Averaging five models does not
  beat the best single model (1.433). Most members are weak, and averaging pulls the predictions even closer to the
  mean (oof_corr falls to 0.112).

## 5. Submission (best non-LLM by CV)

`fit_full()` refits the chosen model on all 120 training titles (no CV split) and returns its 30 test predictions.

In [8]:
best = [k for k in res.index if k != "constant mean"][0]
print("best non-LLM:", best, res.loc[best].round(3).to_dict())

def fit_full(name):
    """Refit the model called `name` on all 120 training titles and return its 30 test predictions.
    The ensemble averages the refit predictions of its member models.
    """
    if name == "ensemble (best per family)":
        return np.mean([fit_full(m) for m in members], 0)
    if name.startswith("fine-tuned"):
        return FineTuned().fit(T[:n], y).predict(T[n:])
    if name.startswith("handcrafted x domain"):
        a = float(name.split("a=")[1]); X = np.hstack([H_int, dom])
        return make_pipeline(StandardScaler(), Ridge(alpha=a)).fit(X[:n], y).predict(X[n:])
    if name.startswith("handcrafted ridge"):
        a = float(name.split("a=")[1]); X = np.hstack([Hs, dom])
        return make_pipeline(StandardScaler(), Ridge(alpha=a)).fit(X[:n], y).predict(X[n:])
    if name.startswith("handcrafted extra-trees"):
        X = np.hstack([Hs, dom])
        return ExtraTreesRegressor(300, min_samples_leaf=8, max_features=0.5, random_state=SEED).fit(X[:n], y).predict(X[n:])
    if name.startswith("mpnet+clipH"):
        return make_pipeline(StandardScaler(), SVR(C=0.3, epsilon=0.5)).fit(X_all[:n], y).predict(X_all[n:])
    k = name.split()[0]; E = EMB[k]; X = np.hstack([E, dom])
    if " ridge " in name: return Ridge(alpha=float(name.split("a=")[1])).fit(X[:n], y).predict(X[n:])
    if " SVR " in name: return make_pipeline(StandardScaler(), SVR(C=float(name.split("C=")[1]), epsilon=0.5)).fit(X[:n], y).predict(X[n:])
    if " kNN " in name: return KNeighborsRegressor(30, metric="cosine", weights="distance").fit(E[:n], y).predict(E[n:])
    raise ValueError(name)

pred = np.clip(fit_full(best), 1, 6)       # no shrink/stretch: CV-chosen s is noise-level
sub = pd.DataFrame({"title_id": test["title_id"], "interest": pred.round(3)})
assert list(sub.columns) == list(sample.columns) and set(sub.title_id) == set(sample.title_id) and len(sub) == 30
sub.to_csv("submission_nonllm.csv", index=False)
print("wrote submission_nonllm.csv | pred range", pred.min().round(2), "-", pred.max().round(2))

best non-LLM: bert SVR C=1 {'mae': 1.433, 'std': 0.172, 'best_s': 1.4, 'mae_at_best_s': 1.428, 'oof_corr': 0.148}


wrote submission_nonllm.csv | pred range 2.08 - 3.7


**Notes on the submission**

* The best model by CV, `bert SVR C=1`, is refit on all 120 training titles and predicts the 30 test titles.
* No shrinkage or stretching is applied (s = 1), because the CV-chosen s is within noise.
* The predictions range from 2.08 to 3.70, a narrow band around the mean (3.13). The model is cautious: it only
  moves a title slightly up or down.
* The `assert` checks that the file has the same columns and title ids as `sample_submission.csv`.

## Summary

* Without an LLM, the best CV MAE is **1.433** (bert SVR), only 0.06 better than the constant model (1.495).
* The useful cues are style cues that differ by domain: long podcast titles, reddit questions, youtube
  hashtags/emoji. Hand-crafted features with domain interactions (1.449) are almost as good as the best encoder,
  and they are easy to interpret.
* Fine-tuning a transformer on 96 titles overfits (1.500). With so few labels, frozen encoders plus a small,
  strongly regularised model are the better choice.
* All non-LLM models stay close to the mean (oof_corr ≤ 0.15). The LLM ratings in llm.ipynb carry a clearly stronger
  signal (ridge on the LLM ratings: CV 1.415).